# Earnings Call EDA + FinBERT — Production Scale

Built for **S&P 500 × 20 years (~1M speaking turns)**.

### Architecture
| Layer | Approach |
|---|---|
| Storage | Parquet partitioned by `ticker` — fast per-firm queries |
| LM scoring | Vectorised regex over full corpus — ~15 min |
| FinBERT scoring | GPU-batched inference with checkpoint/resume — 2–4 hrs on GPU |
| Charts | Always aggregated to call-level or firm-quarter before plotting |
| Caching | Every expensive step writes to disk; re-runs skip completed work |

### Expected input
`calls.csv` — one row per speaking turn with columns:
`transcriptid`, `transcriptcomponentid`, `componentorder`, `transcriptcomponenttypename`,
`transcriptpersonname`, `title`, `speakertypeid`, `componenttext`,
`ticker`, `companyname`, `mostimportantdateutc`, `quarter`, `close_to_open_return`

### Install
```bash
pip install transformers torch pandas pyarrow fastparquet pysentiment2 plotly nltk tqdm
```

---
## 0 · Imports, Config & Paths

In [94]:
import warnings, re, pickle, gc, time
from pathlib import Path

warnings.filterwarnings('ignore')

import numpy  as np
import pandas as pd
from tqdm.notebook import tqdm

import plotly.graph_objects as go
import plotly.express       as px
from plotly.subplots import make_subplots

import nltk
nltk.download('punkt',     quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
from nltk.tokenize import sent_tokenize

# ── Paths ─────────────────────────────────────────────────────────────────────
DATA_DIR    = Path('../exec_transcripts')           # root for all data artefacts
RAW_PATH    = DATA_DIR / 'aggregated_exec_calls.csv' # your input CSV
PARQUET_DIR = DATA_DIR / 'parquet'   # partitioned parquet store
PROC_DIR    = DATA_DIR / 'processed' # scored outputs
CACHE_DIR   = DATA_DIR / 'cache'     # FinBERT checkpoint + LM cache

for p in [PARQUET_DIR, PROC_DIR, CACHE_DIR]:
    p.mkdir(parents=True, exist_ok=True)

# ── Subset flags ──────────────────────────────────────────────────────────────
# Set FINBERT_TICKERS to a list of tickers to restrict FinBERT scoring.
# LM scoring and structural EDA always run on the full corpus.
# None = score all tickers (slow without a GPU)
FINBERT_TICKERS = None        # e.g. ['AAPL','MSFT','NVDA'] or None
FINBERT_YEARS   = None          # e.g. range(2018, 2024) or None

# ── FinBERT batching ──────────────────────────────────────────────────────────
BATCH_SIZE       = 64           # reduce to 32 if OOM on GPU
MAX_TOKENS       = 450          # chunk size; FinBERT hard limit is 512
CHECKPOINT_EVERY = 2_000        # write checkpoint every N rows

# ── Dark theme ────────────────────────────────────────────────────────────────
BG       = '#0d0d14'
BG2      = '#1a1a28'
GOLD     = '#f5a623'
TEAL     = '#4ecdc4'
RED      = '#f87171'
GREEN    = '#34d399'
TEXT     = '#e8e8f0'
MUTED    = '#8888aa'
GRID     = '#1e1e30'
BORDER   = '#2a2a42'
PALETTE  = [GOLD, TEAL, '#a78bfa', GREEN, '#f472b6', '#60a5fa', '#fb923c', RED]
ROLE_COL = {'CEO': GOLD, 'CFO': TEAL, 'Other': '#a78bfa'}

LAYOUT = dict(
    paper_bgcolor=BG, plot_bgcolor=BG,
    font   =dict(family='sans-serif', color=TEXT, size=12),
    legend =dict(bgcolor=BG2, bordercolor=BORDER, borderwidth=1,
                 font=dict(color=MUTED, size=11)),
    xaxis  =dict(gridcolor=GRID, linecolor=BORDER, tickcolor=BORDER,
                 tickfont=dict(color=MUTED)),
    yaxis  =dict(gridcolor=GRID, linecolor='rgba(0,0,0,0)',
                 tickfont=dict(color=MUTED)),
    margin =dict(l=60, r=30, t=60, b=50),
)

def theme(fig, title='', xlab='', ylab='', h=420):
    fig.update_layout(**LAYOUT, height=h,
        title=dict(text=title, font=dict(color=TEXT, size=15)))
    if xlab: fig.update_xaxes(title_text=xlab, title_font=dict(color=MUTED))
    if ylab: fig.update_yaxes(title_text=ylab, title_font=dict(color=MUTED))
    return fig

print('Config ready.')

Config ready.


---
## 1 · Ingest — CSV → Partitioned Parquet

Run once. Subsequent cells load from parquet (fast, per-ticker queries).

In [95]:
PARQUET_DONE = (PARQUET_DIR / '_SUCCESS').exists()

if PARQUET_DONE:
    print('Parquet store already built — skipping ingest.')
else:
    print(f'Reading {RAW_PATH} in chunks...')
    CHUNK = 200_000
    written = 0

    section_map = {
        'Presenter Speech': 'Prepared Remarks',
        'Answer':           'Q&A',
        'Question':         'Q&A',
    }

    def normalise_role(t):
        if not isinstance(t, str): return 'Other'
        t = t.lower()
        if re.search(r'ceo|chief executive|president', t): return 'CEO'
        if re.search(r'cfo|chief financial|finance officer', t): return 'CFO'
        if re.search(r'coo|chief operating', t): return 'COO'
        return 'Other'

    for chunk in tqdm(pd.read_csv(RAW_PATH, chunksize=CHUNK), desc='Chunks'):
        # ── Core transforms ───────────────────────────────────────────────────
        chunk['role']       = chunk['title'].apply(normalise_role)
        chunk['section']    = chunk['transcriptcomponenttypename'] \
                                   .map(section_map).fillna(chunk['transcriptcomponenttypename'])
        # Date column can vary by export; support common names
        date_col = next(
            (c for c in ['mostimportantdateutc', 'mostimportantdateutc_utc', 'call_date', 'date'] if c in chunk.columns),
            None
        )
        if date_col is None:
            raise KeyError(
                "No call date column found. Expected one of: mostimportantdateutc, mostimportantdateutc_utc, call_date, date"
            )

        chunk['call_date']  = pd.to_datetime(chunk[date_col]).dt.normalize()
        chunk['call_year']  = chunk['call_date'].dt.year
        chunk['call_month'] = chunk['call_date'].dt.month
        chunk['quarter']    = pd.to_datetime(chunk['quarter']).dt.to_period('Q').astype(str)

        # ── Text metrics (fast, no NLP) ───────────────────────────────────────
        txt = chunk['componenttext'].fillna('')
        chunk['word_count']   = txt.str.split().str.len().fillna(0).astype(int)
        chunk['char_count']   = txt.str.len().fillna(0).astype(int)
        # Approximate sentence count via punctuation — avoids slow sent_tokenize here
        chunk['sent_count']   = txt.str.count(r'[.!?]+').clip(lower=1)
        chunk['avg_sent_len'] = (chunk['word_count'] / chunk['sent_count']).round(1)

        # ── fastparquet compatibility: avoid ArrowExtensionArray strings ───────
        for c in [
            'transcriptcomponenttypename','transcriptpersonname','title',
            'componenttext','ticker','companyname'
        ]:
            if c in chunk.columns:
                chunk[c] = chunk[c].fillna('').astype(str)

        # ── Write partitioned by ticker (pyarrow; avoids fastparquet string issues) ──
        import pyarrow as pa
        import pyarrow.parquet as pq

        table = pa.Table.from_pandas(chunk, preserve_index=False)
        pq.write_to_dataset(
            table,
            root_path=str(PARQUET_DIR),
            partition_cols=['ticker'],
            compression='snappy',
        )

        written += len(chunk)

    (PARQUET_DIR / '_SUCCESS').touch()
    print(f'Wrote {written:,} rows to {PARQUET_DIR}')

# ── Helper: load subset from parquet ─────────────────────────────────────────
def load_calls(tickers=None, years=None, cols=None):
    """Load from parquet with optional ticker/year filters. Fast for small subsets.
    Rows are unordered; to take the **latest** call for a ticker use
    ``sal.latest_call_turns(load_calls(...), ticker='AAPL')`` after importing salience_dictionary.
    """
    filters = []
    if tickers: filters.append(('ticker', 'in', list(tickers)))
    if years:   filters.append(('call_year', 'in', list(years)))
    df = pd.read_parquet(
        PARQUET_DIR,
        engine='pyarrow',
        filters=filters or None,
        columns=cols,
    )
    return df

# Quick sanity check
sample = load_calls(cols=['ticker','call_year','role','word_count'],tickers=FINBERT_TICKERS)
print(f'Total rows: {len(sample):,}')
print(f'Tickers:    {sample["ticker"].nunique():,}')
print(f'Year range: {sample["call_year"].min()} – {sample["call_year"].max()}')
print(f'Roles:\n{sample["role"].value_counts()}')

Parquet store already built — skipping ingest.
Total rows: 904,458
Tickers:    756
Year range: 2006 – 2024
Roles:
role
CEO      592322
CFO      160943
Other    151193
Name: count, dtype: int64


---
## 2 · Structural EDA — Full Corpus

All charts aggregate to call-level or firm-quarter before plotting.

In [96]:
# ── 2a. Load lightweight columns only ─────────────────────────────────────────
# Include transcriptcomponentid so we can merge LM/FinBERT scores back to turns
eda_cols = ['transcriptid','transcriptcomponentid','ticker','companyname',
            'call_date','call_year','quarter','role','section',
            'word_count','sent_count','avg_sent_len','close_to_open_return']

eda = load_calls(cols=eda_cols)
print(f'EDA frame: {len(eda):,} rows, {eda["transcriptid"].nunique():,} calls')

EDA frame: 904,458 rows, 26,893 calls


In [97]:
# ── 2b. Transcripts per year ───────────────────────────────────────────────────
by_year = (
    eda.groupby('call_year')['transcriptid']
       .nunique()
       .reset_index(name='n_calls')
       .sort_values('call_year')
)

fig = go.Figure(go.Bar(
    x=by_year['call_year'], y=by_year['n_calls'],
    marker_color=GOLD, marker_line_width=0,
    text=by_year['n_calls'], textposition='outside',
    textfont=dict(color=TEXT, size=9),
))
theme(fig, 'Earnings Calls per Year — Full Corpus', 'Year', '# Calls', h=380)
fig.show()

In [98]:
# ── 2c. Coverage heatmap: firm × year (top 60 firms by call count) ────────────
top_tickers = (
    eda.groupby('ticker')['transcriptid'].nunique()
       .nlargest(60).index.tolist()
)

heat = (
    eda[eda['ticker'].isin(top_tickers)]
       .groupby(['ticker','call_year'])['transcriptid']
       .nunique()
       .reset_index(name='n_calls')
)

heat_pivot = heat.pivot(index='ticker', columns='call_year', values='n_calls').fillna(0)

fig = go.Figure(go.Heatmap(
    z            = heat_pivot.values,
    x            = heat_pivot.columns.tolist(),
    y            = heat_pivot.index.tolist(),
    # Plotly doesn't reliably accept 8-digit hex like "#RRGGBBAA"; use rgba() instead
    colorscale   = [
        [0.0, BG2],
        [0.5, 'rgba(245,166,35,0.55)'],
        [1.0, GOLD],
    ],
    hovertemplate = '<b>%{y}</b>  %{x}<br>Calls: %{z}<extra></extra>',
))
theme(fig, 'Coverage Heatmap — Top 60 Firms', 'Year', '', h=700)
fig.update_yaxes(tickfont=dict(size=8, color=MUTED))
fig.show()

In [99]:
# ── 2d. Talk-time share: CEO vs CFO per year ──────────────────────────────────
# Aggregate to year × role — never plot raw rows
talk_time = (
    eda[eda['role'].isin(['CEO','CFO'])]
       .groupby(['call_year','role'])['word_count']
       .mean()
       .reset_index(name='mean_words')
)

fig = go.Figure()
for role, color in [('CEO', GOLD), ('CFO', TEAL)]:
    d = talk_time[talk_time['role'] == role].sort_values('call_year')
    fig.add_trace(go.Scatter(
        x=d['call_year'], y=d['mean_words'].round(0),
        mode='lines+markers', name=role,
        line=dict(color=color, width=2.5),
        marker=dict(color=color, size=6, line=dict(color=BG, width=1.5)),
    ))
theme(fig, 'Avg Words per Turn — CEO vs CFO Over Time', 'Year', 'Mean Words / Turn', h=380)
fig.show()

In [100]:
# ── 2e. Prepared remarks vs Q&A word share over time ─────────────────────────
section_time = (
    eda[eda['section'].isin(['Prepared Remarks','Q&A'])]
       .groupby(['call_year','section'])['word_count']
       .sum()
       .reset_index()
)
section_pct = section_time.copy()
total_by_year = section_time.groupby('call_year')['word_count'].transform('sum')
section_pct['pct'] = section_time['word_count'] / total_by_year * 100

fig = go.Figure()
colors = {'Prepared Remarks': TEAL, 'Q&A': GOLD}
for sect in ['Prepared Remarks', 'Q&A']:
    d = section_pct[section_pct['section'] == sect].sort_values('call_year')
    fig.add_trace(go.Bar(
        x=d['call_year'], y=d['pct'].round(1),
        name=sect, marker_color=colors[sect],
    ))
theme(fig, 'Section Word Share Over Time', 'Year', '% of Total Words', h=380)
fig.update_layout(barmode='stack')
fig.show()

---
## 3 · LM Dictionary Scoring — Full Corpus

Vectorised over chunks — no row-by-row `apply()`. Scores written to parquet once and reloaded on re-runs.

In [101]:
# ---- LM caching strategy ----
# Cache expensive LM scoring per-ticker so reruns can resume/skip by ticker.
# Do NOT cache aggregated charts/frames (recompute depending on subset vs full).

LM_TICKER_DIR = PROC_DIR / 'lm_by_ticker'
LM_TICKER_DIR.mkdir(parents=True, exist_ok=True)

def _available_tickers_from_parquet(root: Path):
    # expects hive partitions like root/ticker=MA/...
    out = []
    for p in root.glob('ticker=*'):
        if p.is_dir():
            out.append(p.name.split('=', 1)[1])
    return sorted(out)

def _as_list(x):
    if x is None:
        return None
    if isinstance(x, str):
        return [x]
    return list(x)

# ── LM word lists / scorer ───────────────────────────────────────────────────
try:
    import pysentiment2 as ps
    _lm = ps.LM()
    USE_PS2 = True
    print('Using pysentiment2 LM dictionary.')
except ImportError:
    USE_PS2 = False
    _lm = None
    print('pysentiment2 not found — using built-in word sets.'
          '  pip install pysentiment2 for the full LM list.')

_POS = {'achieve','achievement','best','better','beneficial','bonus','confidence',
        'deliver','driven','effective','efficient','excellent','exceptional',
        'expand','growth','improve','improvement','increase','innovative','leader',
        'opportunity','outperform','positive','profit','record','robust','solid',
        'strong','succeed','success','superior','sustainable','value'}
_NEG = {'adverse','burden','challenge','concern','constrain','contraction',
        'decline','decrease','default','deficit','delay','difficult','disappointing',
        'dispute','drop','fail','headwind','impaired','impairment','inadequate',
        'insufficient','issue','loss','miss','negative','penalty','pressure',
        'problem','reduce','risk','shortfall','uncertain','underperform',
        'volatility','weak','weakness'}
_UNC = {'approximately','assume','belief','believe','contingent','depend',
        'estimate','expect','forecast','guidance','hope','indicate','likely',
        'may','might','outlook','pending','plan','possible','potentially',
        'predict','project','seem','should','suppose','uncertain','unclear',
        'unknown','unless','volatile','whether'}


def _score_lm_series(texts: pd.Series) -> pd.DataFrame:
    rows = []
    for text in texts:
        if not isinstance(text, str) or len(text.strip()) < 5:
            rows.append({'lm_pos':0,'lm_neg':0,'lm_unc':0,
                         'lm_net':0.0,'lm_unc_pct':0.0,'lm_n_words':0})
            continue

        if USE_PS2:
            tokens = _lm.tokenize(text)
            sc     = _lm.get_score(tokens)
            pos    = sc.get('Positive', 0)
            neg    = sc.get('Negative', 0)
            unc    = sc.get('Uncertainty', 0)
            n      = max(len(tokens), 1)
        else:
            words = re.findall(r'[a-z]+', text.lower())
            pos   = sum(w in _POS for w in words)
            neg   = sum(w in _NEG for w in words)
            unc   = sum(w in _UNC for w in words)
            n     = max(len(words), 1)

        tot = max(pos + neg, 1)
        rows.append({
            'lm_pos':     pos,
            'lm_neg':     neg,
            'lm_unc':     unc,
            'lm_n_words': n,
            'lm_net':     round((pos - neg) / tot, 5),
            'lm_unc_pct': round(unc / n, 5),
        })
    return pd.DataFrame(rows)


def lm_score_ticker(ticker: str):
    out_path = LM_TICKER_DIR / f'ticker={ticker}.parquet'
    if out_path.exists():
        print(f'[{ticker}] LM already cached — skipping.')
        return

    id_cols = ['transcriptid','transcriptcomponentid','ticker','call_year']
    df = load_calls(
        tickers=[ticker],
        years=None,
        cols=id_cols + ['componenttext'],
    ).reset_index(drop=True)

    print(f'[{ticker}] LM turns: {len(df):,}')
    scores = _score_lm_series(df['componenttext'].fillna(''))
    out = pd.concat([df[id_cols], scores], axis=1)
    out.to_parquet(out_path, index=False, engine='pyarrow')
    print(f'[{ticker}] wrote {len(out):,} rows → {out_path}')


# ---- Entry point ----
# Mirror FINBERT_TICKERS restriction for LM caching convenience.
req = _as_list(FINBERT_TICKERS)
if req is None:
    tickers_to_run = _available_tickers_from_parquet(PARQUET_DIR)
else:
    tickers_to_run = req

print(f'LM tickers to run: {len(tickers_to_run):,}')
for t in tickers_to_run:
    lm_score_ticker(t)

# Combine per-ticker outputs computed so far
lm_df = pd.read_parquet(LM_TICKER_DIR, engine='pyarrow')
print(f'Loaded combined LM frame: {len(lm_df):,} rows from {LM_TICKER_DIR}')

print(lm_df[['lm_net','lm_unc_pct']].describe().round(4))

Using pysentiment2 LM dictionary.
LM tickers to run: 756
[A] LM already cached — skipping.
[AA] LM already cached — skipping.
[AABA] LM already cached — skipping.
[AAL] LM already cached — skipping.
[AAP] LM already cached — skipping.
[AAPL] LM already cached — skipping.
[AAXN] LM already cached — skipping.
[ABBV.WI] LM already cached — skipping.
[ABC] LM already cached — skipping.
[ABMD] LM already cached — skipping.
[ABNB] LM already cached — skipping.
[ABT] LM already cached — skipping.
[ACAS] LM already cached — skipping.
[ADBE] LM already cached — skipping.
[ADI] LM already cached — skipping.
[ADM] LM already cached — skipping.
[ADP] LM already cached — skipping.
[ADSK] LM already cached — skipping.
[ADT.WI] LM already cached — skipping.
[AEE] LM already cached — skipping.
[AEP] LM already cached — skipping.
[AES] LM already cached — skipping.
[AET] LM already cached — skipping.
[AFL] LM already cached — skipping.
[AGL] LM already cached — skipping.
[AGN] LM already cached — skipp

In [102]:
lm_df

,transcriptid,transcriptcomponentid,ticker,call_year,lm_pos,lm_neg,lm_unc,lm_n_words,lm_net,lm_unc_pct
0,1804833,70658628,A,2019,0.0,0.0,0,9,0.00000,0.0
1,1804833,70658592,A,2019,1.0,0.0,0,5,1.00000,0.0
2,1804833,70658612,A,2019,0.0,0.0,0,7,0.00000,0.0
3,1804833,70658596,A,2019,2.0,1.0,0,25,0.33333,0.0
4,1804833,70658600,A,2019,0.0,1.0,0,47,-1.00000,0.0
...,...,...,...,...,...,...,...,...,...,...
903541,3213797,114400235,ZTS,2024,4.0,8.0,0,89,-0.33333,0.0
903542,3213797,114400239,ZTS,2024,6.0,6.0,0,72,0.00000,0.0
903543,3213797,114400203,ZTS,2024,63.0,13.0,0,681,0.65789,0.0
903544,3213797,114400207,ZTS,2024,4.0,1.0,0,125,0.60000,0.0


In [103]:
# ── 3a. LM net sentiment by year — aggregated ─────────────────────────────────
# Join role & section from eda frame (lightweight — no text column)
# lm_df already contains ticker and call_year (from parquet). Only bring speaker metadata from eda.
lm_eda = lm_df.merge(
    eda[['transcriptid','transcriptcomponentid','quarter',
         'role','section','close_to_open_return']],
    on=['transcriptid','transcriptcomponentid'], how='left'
)

# Aggregate to call × role level first
lm_call = (
    lm_eda[lm_eda['role'].isin(['CEO','CFO'])]
       .groupby(['transcriptid','call_year','ticker','role','close_to_open_return'])
       .agg(
           mean_lm_net  = ('lm_net', 'mean'),
           mean_lm_unc  = ('lm_unc_pct', 'mean'),
           total_words  = ('lm_n_words', 'sum'),
       )
       .reset_index()
)

# Year-level ribbon
lm_yr = (
    lm_call
       .groupby(['call_year','role'])['mean_lm_net']
       .agg(['mean', lambda x: x.quantile(0.25), lambda x: x.quantile(0.75)])
       .rename(columns={'mean':'mean', '<lambda_0>':'q25', '<lambda_1>':'q75'})
       .reset_index()
       .sort_values('call_year')
)

fig = go.Figure()
BAND_FILL = {
    'CEO': 'rgba(245,166,35,0.13)',  # GOLD with alpha
    'CFO': 'rgba(78,205,196,0.13)',  # TEAL with alpha
}
for role, color in [('CEO', GOLD), ('CFO', TEAL)]:
    d = lm_yr[lm_yr['role'] == role]
    # IQR band
    fig.add_trace(go.Scatter(
        x=pd.concat([d['call_year'], d['call_year'][::-1]]),
        y=pd.concat([d['q75'], d['q25'][::-1]]),
        fill='toself', fillcolor=BAND_FILL[role],
        line=dict(width=0), showlegend=False, hoverinfo='skip',
    ))
    # Mean line
    fig.add_trace(go.Scatter(
        x=d['call_year'], y=d['mean'].round(4),
        mode='lines+markers', name=f'{role} mean',
        line=dict(color=color, width=2.5),
        marker=dict(color=color, size=5, line=dict(color=BG, width=1.5)),
    ))

fig.add_hline(y=0, line_dash='dot', line_color=BORDER, line_width=1)
theme(fig, 'LM Net Sentiment Over Time — CEO vs CFO  (mean ± IQR)',
      'Year', 'Mean LM Net Sentiment', h=420)
fig.show()

In [104]:
# ── 3b. LM uncertainty over time ──────────────────────────────────────────────
unc_yr = (
    lm_call
       .groupby(['call_year','role'])['mean_lm_unc']
       .mean()
       .reset_index()
       .sort_values('call_year')
)

fig = go.Figure()
for role, color in [('CEO', GOLD), ('CFO', TEAL)]:
    d = unc_yr[unc_yr['role'] == role]
    fig.add_trace(go.Scatter(
        x=d['call_year'], y=d['mean_lm_unc'].round(4),
        mode='lines+markers', name=role,
        line=dict(color=color, width=2.5),
        marker=dict(color=color, size=5, line=dict(color=BG, width=1.5)),
    ))

theme(fig, 'LM Uncertainty % Over Time — CEO vs CFO',
      'Year', 'Mean Uncertainty (% of words)', h=380)
fig.show()

---
## 4 · FinBERT Scoring — Batched, GPU-Accelerated, Checkpointed

Scores are written to a checkpoint file every `CHECKPOINT_EVERY` rows and to a final parquet on completion.  
**Interrupting and re-running this cell resumes from the last checkpoint automatically.**

Set `FINBERT_TICKERS` / `FINBERT_YEARS` at the top of the notebook to restrict the scoring scope.

In [105]:
from transformers import pipeline, AutoTokenizer
import torch

# FinBERT outputs are cached per-ticker under PROC_DIR/finbert_by_ticker/
MODEL_NAME = 'ProsusAI/finbert'

# ── Load model ────────────────────────────────────────────────────────────────
print(f'Loading {MODEL_NAME}...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
finbert   = pipeline(
    'text-classification',
    model     = MODEL_NAME,
    tokenizer = tokenizer,
    top_k     = None,
    device    = 0 if torch.cuda.is_available() else -1,
    batch_size= BATCH_SIZE,
)
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
    print(f'Estimated throughput: ~{BATCH_SIZE * 3_600 // 512:,} turns/hr')
else:
    print('Running on CPU — consider restricting FINBERT_TICKERS.')

Loading ProsusAI/finbert...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


CUDA available: False
Running on CPU — consider restricting FINBERT_TICKERS.


In [106]:
# ── Chunking helper ───────────────────────────────────────────────────────────
def chunk_text(text, max_tokens=MAX_TOKENS):
    """Split text on sentence boundaries into ≤max_tokens chunks."""
    if not isinstance(text, str) or len(text.strip()) < 5:
        return []
    sentences = sent_tokenize(text)
    chunks, current, current_len = [], [], 0
    for sent in sentences:
        n = len(tokenizer.tokenize(sent))
        if current_len + n > max_tokens and current:
            chunks.append(' '.join(current))
            current, current_len = [sent], n
        else:
            current.append(sent)
            current_len += n
    if current:
        chunks.append(' '.join(current))
    return chunks or [text[:1500]]

def aggregate_chunk_scores(chunk_results):
    """Average pos/neg/neu probabilities across chunks for one turn."""
    pos = neg = neu = 0.0
    for result in chunk_results:
        s = {r['label'].lower(): r['score'] for r in result}
        pos += s.get('positive', 0)
        neg += s.get('negative', 0)
        neu += s.get('neutral',  0)
    n = len(chunk_results)
    return {
        'fb_positive': round(pos / n, 5),
        'fb_negative': round(neg / n, 5),
        'fb_neutral':  round(neu / n, 5),
    }

print('Chunking helpers ready.')

Chunking helpers ready.


In [108]:
# ---- FinBERT caching strategy ----
# Cache expensive scoring per-ticker so you can unrestrict FINBERT_TICKERS
# without needing (or trying) to checkpoint the entire full-corpus run.
# Aggregations later in the notebook should be recomputed (not cached) when you
# run subsets.

FB_TICKER_DIR = PROC_DIR / 'finbert_by_ticker'
FB_TICKER_DIR.mkdir(parents=True, exist_ok=True)

def _available_tickers_from_parquet(root: Path):
    # expects hive partitions like root/ticker=MA/...
    out = []
    for p in root.glob('ticker=*'):
        if p.is_dir():
            out.append(p.name.split('=', 1)[1])
    return sorted(out)

def _as_list(x):
    if x is None:
        return None
    if isinstance(x, str):
        return [x]
    return list(x)

def finbert_score_ticker(ticker: str):
    out_path = FB_TICKER_DIR / f'ticker={ticker}.parquet'
    if out_path.exists():
        print(f'[{ticker}] FinBERT already cached — skipping.')
        return

    id_cols = ['transcriptid','transcriptcomponentid','ticker','call_year']
    to_score = load_calls(
        tickers=[ticker],
        years=list(FINBERT_YEARS) if FINBERT_YEARS else None,
        cols=id_cols + ['componenttext'],
    ).reset_index(drop=True)

    print(f'[{ticker}] turns to score: {len(to_score):,}')

    # Use an in-memory checkpoint per ticker; write final parquet for this ticker.
    scored_dict = {}

    # ---- Build chunk plan (same logic as before) ----
    def chunk_text(text: str, max_tokens=MAX_TOKENS):
        if not isinstance(text, str) or not text.strip():
            return []
        words = text.split()
        # approximate token->word; conservative chunking
        step = max(50, int(max_tokens * 0.75))
        return [' '.join(words[i:i+step]) for i in range(0, len(words), step)]

    chunk_texts = []   # list[tuple[row_idx, chunk_idx, text]]
    row_chunk_results = {}

    for row_idx, text in enumerate(to_score['componenttext'].fillna('')):
        chunks = chunk_text(text)
        if not chunks:
            scored_dict[row_idx] = {'fb_positive': None, 'fb_negative': None, 'fb_neutral': None}
            continue
        row_chunk_results[row_idx] = []
        for j, ch in enumerate(chunks):
            chunk_texts.append((row_idx, j, ch))

    def aggregate_chunk_scores(chunk_res_list):
        # chunk_res_list: list of dicts with keys fb_positive/fb_negative/fb_neutral
        vals = {'fb_positive': [], 'fb_negative': [], 'fb_neutral': []}
        for d in chunk_res_list:
            for k in vals:
                if d.get(k) is not None:
                    vals[k].append(d[k])
        if all(len(vals[k]) == 0 for k in vals):
            return {'fb_positive': None, 'fb_negative': None, 'fb_neutral': None}
        return {k: float(np.mean(vals[k])) if len(vals[k]) else None for k in vals}

    # ---- Batched inference over chunks ----
    t0 = time.time()
    scored_chunks = 0

    for i in tqdm(range(0, len(chunk_texts), BATCH_SIZE), desc=f'FinBERT [{ticker}]'):
        batch = chunk_texts[i:i+BATCH_SIZE]
        texts = [t for (_, _, t) in batch]
        # Ensure we never exceed model max length (512) even if our chunking heuristic is off
        outputs = finbert(texts, truncation=True, max_length=512)

        for (row_idx, _, _), out in zip(batch, outputs):
            # out is list of label/score dicts
            label_map = {d['label'].lower(): float(d['score']) for d in out}
            row_chunk_results[row_idx].append({
                'fb_positive': label_map.get('positive'),
                'fb_negative': label_map.get('negative'),
                'fb_neutral':  label_map.get('neutral'),
            })

        scored_chunks += len(batch)

    # Consolidate per-row
    for row_idx, chunk_res in row_chunk_results.items():
        scored_dict[row_idx] = aggregate_chunk_scores(chunk_res)

    scores_df = pd.DataFrame([
        {'row_idx': k, **v} for k, v in scored_dict.items()
    ]).set_index('row_idx')

    fb_df_t = to_score[id_cols].copy().join(scores_df)
    fb_df_t['fb_net'] = fb_df_t['fb_positive'] - fb_df_t['fb_negative']

    probs = fb_df_t[['fb_positive','fb_negative','fb_neutral']]
    all_na = probs.isna().all(axis=1)
    fb_df_t['fb_label'] = 'na'
    fb_df_t.loc[~all_na, 'fb_label'] = (
        probs.loc[~all_na].fillna(-1.0).idxmax(axis=1).str.replace('fb_', '', regex=False)
    )

    fb_df_t.to_parquet(out_path, index=False, engine='pyarrow')
    elapsed = time.time() - t0
    print(f'[{ticker}] wrote {len(fb_df_t):,} rows → {out_path}  ({elapsed/60:.1f} min)')

# ---- Entry point ----
req = _as_list(FINBERT_TICKERS)
if req is None:
    tickers_to_run = _available_tickers_from_parquet(PARQUET_DIR)
else:
    tickers_to_run = req

print(f'FinBERT tickers to run: {len(tickers_to_run):,}')
for t in tickers_to_run:
    finbert_score_ticker(t)

# Combine all per-ticker outputs (computed so far) into one frame for downstream plots
fb_df = pd.read_parquet(FB_TICKER_DIR, engine='pyarrow')
print(f'Loaded combined FinBERT frame: {len(fb_df):,} rows from {FB_TICKER_DIR}')


print(fb_df[['fb_positive','fb_negative','fb_neutral','fb_net']].describe().round(4))

FinBERT tickers to run: 756
[A] FinBERT already cached — skipping.
[AA] turns to score: 735


FinBERT [AA]:   0%|          | 0/20 [00:00<?, ?it/s]

[AA] wrote 735 rows → ../exec_transcripts/processed/finbert_by_ticker/ticker=AA.parquet  (0.9 min)
[AABA] turns to score: 627


FinBERT [AABA]:   0%|          | 0/16 [00:00<?, ?it/s]

[AABA] wrote 627 rows → ../exec_transcripts/processed/finbert_by_ticker/ticker=AABA.parquet  (0.7 min)
[AAL] turns to score: 1,737


...

[AYI] wrote 184 rows → ../exec_transcripts/processed/finbert_by_ticker/ticker=AYI.parquet  (0.2 min)
[AZO] turns to score: 1,579


FinBERT [AZO]:   0%|          | 0/37 [00:00<?, ?it/s]

[AZO] wrote 1,579 rows → ../exec_transcripts/processed/finbert_by_ticker/ticker=AZO.parquet  (1.7 min)
[BA] turns to score: 2,017


FinBERT [BA]:   0%|          | 0/42 [00:00<?, ?it/s]

KeyboardInterrupt: 

---
## 5 · FinBERT Analysis — Aggregated Charts

In [141]:
fb_df = pd.read_parquet(FB_TICKER_DIR, engine='pyarrow')
print(f'Loaded combined FinBERT frame: {len(fb_df):,} rows from {FB_TICKER_DIR}')


print(fb_df[['fb_positive','fb_negative','fb_neutral','fb_net']].describe().round(4))

Loaded combined FinBERT frame: 266,156 rows from ../exec_transcripts/processed/finbert_by_ticker
       fb_positive  fb_negative   fb_neutral       fb_net
count  266156.0000  266156.0000  266156.0000  266156.0000
mean        0.2984       0.1038       0.5977       0.1946
std         0.2762       0.1910       0.2989       0.3690
min         0.0067       0.0063       0.0101      -0.9679
25%         0.0765       0.0158       0.3281       0.0169
50%         0.1802       0.0274       0.7115       0.1273
75%         0.4790       0.0744       0.8615       0.4125
max         0.9634       0.9747       0.9563       0.9475


In [137]:
fb_df

,transcriptid,transcriptcomponentid,ticker,call_year,fb_positive,fb_negative,fb_neutral,fb_net,fb_label
0,1804833,70658628,A,2019,0.548661,0.035194,0.416145,0.513466,positive
1,1804833,70658592,A,2019,0.053089,0.014786,0.932125,0.038302,neutral
2,1804833,70658612,A,2019,0.057617,0.061813,0.880570,-0.004196,neutral
3,1804833,70658596,A,2019,0.855000,0.007454,0.137546,0.847546,positive
4,1804833,70658600,A,2019,0.763015,0.025618,0.211367,0.737396,positive
...,...,...,...,...,...,...,...,...,...
234091,407577,19243218,MA,2013,0.110925,0.042936,0.846139,0.067989,neutral
234092,407577,19243226,MA,2013,0.030594,0.492544,0.476861,-0.461950,negative
234093,407577,19243234,MA,2013,0.139054,0.062441,0.798505,0.076613,neutral
234094,407577,19243242,MA,2013,0.156717,0.017736,0.825547,0.138982,neutral


In [142]:
# ── Join FinBERT scores with metadata ─────────────────────────────────────────
# fb_df already carries ticker/call_year from parquet. Only bring speaker metadata from eda.
meta_cols = ['transcriptid','transcriptcomponentid','quarter',
             'role','section','close_to_open_return','word_count']

fb_meta = fb_df.merge(
    eda[meta_cols],
    on=['transcriptid','transcriptcomponentid'],
    how='left',
)

# ── Aggregate to call × role level (the plotting unit) ────────────────────────
fb_call = (
    fb_meta[fb_meta['role'].isin(['CEO','CFO'])]
       .groupby(['transcriptid','ticker','call_year','quarter',
                 'role','close_to_open_return'])
       .agg(
           mean_fb_net  = ('fb_net', 'mean'),
           mean_fb_pos  = ('fb_positive', 'mean'),
           mean_fb_neg  = ('fb_negative', 'mean'),
           n_turns      = ('fb_net', 'count'),
           pct_positive = ('fb_label', lambda x: (x == 'positive').mean()),
           pct_negative = ('fb_label', lambda x: (x == 'negative').mean()),
       )
       .reset_index()
)

print(f'Call-level frame: {len(fb_call):,} rows  '
      f'({fb_call["ticker"].nunique()} tickers, '
      f'{fb_call["call_year"].nunique()} years)')

Call-level frame: 12,345 rows  (220 tickers, 19 years)


In [143]:
# ── 5a. FinBERT net sentiment ribbon — by year ────────────────────────────────
fb_yr = (
    fb_call
       .groupby(['call_year','role'])['mean_fb_net']
       .agg(['mean',
             lambda x: x.quantile(0.45),
             lambda x: x.quantile(0.65)])
       .rename(columns={'mean':'mean','<lambda_0>':'q25','<lambda_1>':'q75'})
       .reset_index()
       .sort_values('call_year')
)

fig = go.Figure()
BAND_FILL = {
    'CEO': 'rgba(245,166,35,0.13)',  # GOLD with alpha
    'CFO': 'rgba(78,205,196,0.13)',  # TEAL with alpha
}


for role, color in [('CEO', GOLD), ('CFO', TEAL)]:
    d = fb_yr[fb_yr['role'] == role]
    fig.add_trace(go.Scatter(
        x=pd.concat([d['call_year'], d['call_year'][::-1]]),
        y=pd.concat([d['q75'], d['q25'][::-1]]),
        fill='toself', fillcolor=BAND_FILL[role],
        line=dict(width=0), showlegend=False, hoverinfo='skip',
    ))
    fig.add_trace(go.Scatter(
        x=d['call_year'], y=d['mean'].round(4),
        mode='lines+markers', name=f'{role} mean',
        line=dict(color=color, width=2.5),
        marker=dict(color=color, size=5, line=dict(color=BG, width=1.5)),
    ))

fig.add_hline(y=0, line_dash='dot', line_color=BORDER, line_width=1)
theme(fig, 'FinBERT Net Sentiment Over Time — CEO vs CFO  (mean ± IQR)',
      'Year', 'Mean FinBERT Net (pos − neg)', h=420)
fig.show()

In [144]:
# ── 5b. LM vs FinBERT cross-validation — call level ──────────────────────────
# Merge call-level aggregates (no raw text needed)
lm_call_agg = (
    lm_eda[lm_eda['role'].isin(['CEO','CFO'])]
       .groupby(['transcriptid','ticker','call_year','role'])
       .agg(mean_lm_net=('lm_net','mean'), mean_lm_unc=('lm_unc_pct','mean'))
       .reset_index()
)

compare = fb_call.merge(
    lm_call_agg, on=['transcriptid','ticker','call_year','role'], how='inner'
)

from scipy.stats import pearsonr
r, p = pearsonr(compare['mean_lm_net'], compare['mean_fb_net'])

# Sample for plotting if large (10k points is fine; 1M is not)
plot_df = compare.sample(min(10_000, len(compare)), random_state=42)

fig = go.Figure()
for role, color in [('CEO', GOLD), ('CFO', TEAL)]:
    d = plot_df[plot_df['role'] == role]
    fig.add_trace(go.Scatter(
        x=d['mean_lm_net'].round(4), y=d['mean_fb_net'].round(4),
        mode='markers', name=role,
        marker=dict(color=color, size=4, opacity=0.5,
                    line=dict(width=0)),
        hovertemplate='<b>%{customdata[0]}</b>  %{customdata[1]}<br>'
                      'LM: %{x:.3f}  FinBERT: %{y:.3f}<extra></extra>',
        customdata=d[['ticker','call_year']].values,
    ))

lim = max(abs(plot_df[['mean_lm_net','mean_fb_net']].values).max() + 0.05, 0.1)
fig.add_trace(go.Scatter(
    x=[-lim, lim], y=[-lim, lim],
    mode='lines', line=dict(color=BORDER, dash='dash', width=1),
    showlegend=False, hoverinfo='skip',
))
fig.add_hline(y=0, line_dash='dot', line_color=BORDER, line_width=0.8)
fig.add_vline(x=0, line_dash='dot', line_color=BORDER, line_width=0.8)

theme(fig,
    f'LM vs FinBERT Net Sentiment — Call Level  (r = {r:.3f}, p = {p:.2e})'
    + (f'  [sampled {len(plot_df):,}/{len(compare):,}]' if len(compare) > 10_000 else ''),
    'Mean LM Net', 'Mean FinBERT Net', h=480)
fig.show()

print(f'Pearson r: {r:.3f}  p: {p:.2e}')
print(f'Sign disagreements: '
      f"{(np.sign(compare['mean_lm_net']) != np.sign(compare['mean_fb_net'])).mean():.1%}")

Pearson r: 0.506  p: 0.00e+00
Sign disagreements: 26.3%


---
## 6 · CEO / CFO Divergence Signal — Cross-Firm, Multi-Year

In [145]:
# ── 6a. Compute within-call CEO/CFO divergence ────────────────────────────────
# Pivot fb_call to wide format: one row per call with CEO and CFO columns
div_wide = (
    fb_call[['transcriptid','ticker','call_year','quarter',
              'role','mean_fb_net','mean_fb_neg','close_to_open_return']]
       .pivot_table(
           index   = ['transcriptid','ticker','call_year','quarter',
                       'close_to_open_return'],
           columns = 'role',
           values  = ['mean_fb_net','mean_fb_neg'],
       )
)
div_wide.columns = ['_'.join(c).lower() for c in div_wide.columns]
div_wide = div_wide.reset_index()

# Only keep calls where we have both CEO and CFO
div_wide = div_wide.dropna(subset=['mean_fb_net_ceo','mean_fb_net_cfo'])

div_wide['ceo_cfo_net_delta'] = div_wide['mean_fb_net_ceo'] - div_wide['mean_fb_net_cfo']
div_wide['ceo_cfo_abs_div']   = div_wide['ceo_cfo_net_delta'].abs()
div_wide['ceo_more_positive'] = div_wide['ceo_cfo_net_delta'] > 0

print(f'Calls with both CEO & CFO scores: {len(div_wide):,}')
print(div_wide[['ceo_cfo_net_delta','ceo_cfo_abs_div',
                 'close_to_open_return']].describe().round(4))

Calls with both CEO & CFO scores: 4,336
       ceo_cfo_net_delta  ceo_cfo_abs_div  close_to_open_return
count          4336.0000        4336.0000             4336.0000
mean              0.0481           0.1675                0.0007
std               0.2162           0.1449                0.0327
min              -0.8945           0.0000               -0.3880
25%              -0.0761           0.0615               -0.0060
50%               0.0485           0.1307                0.0003
75%               0.1720           0.2351                0.0074
max               1.2715           1.2715                0.2636


In [146]:
# ── 6b. Divergence vs close-to-open return scatter ────────────────────────────
plot_d = div_wide.sample(min(5_000, len(div_wide)), random_state=42)

fig = go.Figure(go.Scatter(
    x    = plot_d['ceo_cfo_abs_div'].round(4),
    y    = plot_d['close_to_open_return'].round(5),
    mode = 'markers',
    marker = dict(
        color   = plot_d['ceo_cfo_net_delta'],
        colorscale = [[0, TEAL], [0.5, BORDER], [1, GOLD]],
        cmin    = plot_d['ceo_cfo_net_delta'].quantile(0.05),
        cmax    = plot_d['ceo_cfo_net_delta'].quantile(0.95),
        size    = 4,
        opacity = 0.55,
        colorbar= dict(
            title=dict(text='CEO − CFO net', font=dict(color=MUTED, size=11)),
            tickfont=dict(color=MUTED),
            bgcolor=BG2,
            bordercolor=BORDER,
            borderwidth=1,
        ),
    ),
    hovertemplate = '<b>%{customdata[0]}</b>  %{customdata[1]}<br>'
                   'Abs divergence: %{x:.3f}<br>Return: %{y:.4f}<extra></extra>',
    customdata = plot_d[['ticker','call_year']].values,
))

fig.add_hline(y=0, line_dash='dot', line_color=BORDER, line_width=0.8)
theme(fig,
    'CEO/CFO Divergence vs Close-to-Open Return'
    + (f'  [sampled {len(plot_d):,}/{len(div_wide):,}]' if len(div_wide) > 5_000 else ''),
    '|CEO − CFO| FinBERT Net', 'Close-to-Open Return', h=500)
fig.show()

In [147]:
# ── 6c. Divergence quintile sort → mean return ────────────────────────────────
div_wide['div_quintile'] = pd.qcut(
    div_wide['ceo_cfo_abs_div'], q=5,
    labels=['Q1\n(low div)', 'Q2', 'Q3', 'Q4', 'Q5\n(high div)']
)

quint = (
    div_wide.groupby('div_quintile')['close_to_open_return']
       .agg(['mean','sem','count'])
       .reset_index()
)
quint['ci95'] = quint['sem'] * 1.96

fig = go.Figure(go.Bar(
    x           = quint['div_quintile'].astype(str),
    y           = quint['mean'].round(5),
    error_y     = dict(type='data', array=quint['ci95'], color=MUTED,
                       thickness=1.5, width=6),
    marker_color= [TEAL if v >= 0 else RED for v in quint['mean']],
    text        = quint['count'].apply(lambda n: f'n={n:,}'),
    textposition= 'outside',
    textfont    = dict(color=MUTED, size=9),
    hovertemplate = '<b>%{x}</b><br>Mean return: %{y:.5f}<br>n: %{text}<extra></extra>',
))

fig.add_hline(y=0, line_dash='dot', line_color=BORDER, line_width=1)
theme(fig,
    'CEO/CFO Divergence Quintile Sort → Mean Close-to-Open Return',
    'Divergence Quintile', 'Mean Close-to-Open Return', h=420)
fig.show()

print(quint[['div_quintile','mean','count']].round(6).to_string(index=False))

  div_quintile      mean  count
 Q1\n(low div)  0.001685    868
            Q2  0.000196    867
            Q3  0.001228    867
            Q4 -0.000096    867
Q5\n(high div)  0.000721    867


In [148]:
# ── 6d. Average divergence over time ──────────────────────────────────────────
div_yr = (
    div_wide.groupby('call_year')['ceo_cfo_abs_div']
       .agg(['mean','median'])
       .reset_index()
       .sort_values('call_year')
)

fig = go.Figure()
for col, color, name in [
    ('mean',   GOLD, 'Mean divergence'),
    ('median', TEAL, 'Median divergence'),
]:
    fig.add_trace(go.Scatter(
        x=div_yr['call_year'], y=div_yr[col].round(4),
        mode='lines+markers', name=name,
        line=dict(color=color, width=2.5),
        marker=dict(color=color, size=5, line=dict(color=BG, width=1.5)),
    ))

theme(fig, 'CEO/CFO Language Divergence Over Time',
      'Year', '|CEO − CFO| FinBERT Net', h=380)
fig.show()

---
## 7 · Save Final Outputs

In [149]:
# ── Load event-study returns for transcript join (inspect columns first) ──────
from pathlib import Path
import pandas as pd

EVENT_PATH = Path('../analytical_ml_finance/aggregated_all_years_updated_event.csv')

# Read a small sample to verify schema (file is very large)
# NOTE: set dtype_backend='numpy_nullable' if you want lighter memory usage
sample = pd.read_csv(EVENT_PATH, nrows=5)
print('Event file columns:')
print(sample.columns.tolist())
print('\nSample rows:')
display(sample)

# Heuristic: keep transcriptid + any columns that look like event-day returns
# (we'll confirm exact names after you review the column list)
col_lower = {c: c.lower() for c in sample.columns}

def looks_like_event_return(col: str) -> bool:
    s = col.lower().replace(' ', '')
    # examples: t0, t-1, t+1, t1, ret_t0, r_t-1, etc.
    return (
        s in {'t0','t-1','t1','t+1','t-2','t+2','t-3','t+3'}
        or s.startswith('t-') or s.startswith('t+')
        or s.startswith('ret_t') or s.startswith('r_t')
    )

return_cols = [c for c in sample.columns if looks_like_event_return(c)]

# transcriptid column name check
id_candidates = [c for c in sample.columns if c.lower() == 'transcriptid']
if not id_candidates:
    print('\nNo transcriptid column found in sample. Candidate ID-like cols:')
    print([c for c in sample.columns if 'transcript' in c.lower() or c.lower().endswith('id')])
else:
    id_col = id_candidates[0]
    keep_cols = [id_col] + return_cols
    print('\nDetected return-like columns:')
    print(return_cols)
    print('\nWill retain these columns (pending your confirmation):')
    print(keep_cols)
    event_small = sample[keep_cols]
    display(event_small)

# Next step: join to existing divergence quintiles and do event-study lines
# Assumes you've already built div_wide with transcript-level divergence and div_quintile.

if 'div_wide' not in globals():
    raise NameError("Expected div_wide to exist (transcript-level divergence frame). Run the divergence section first.")

# Use the same quintile labels already defined earlier in the notebook
quint_map = div_wide[['transcriptid','div_quintile']].dropna().copy()
quint_map['transcriptid'] = pd.to_numeric(quint_map['transcriptid'], errors='coerce').astype('Int64')
quint_map = quint_map.dropna(subset=['transcriptid'])

# Stream the huge CSV in chunks and aggregate returns by quintile
CHUNK_ROWS = 250_000

# Running sums and counts per quintile per event column
sum_by_q = None
cnt_by_q = None

usecols = keep_cols  # transcriptid + identified return columns

for chunk in pd.read_csv(EVENT_PATH, usecols=usecols, chunksize=CHUNK_ROWS):
    chunk[id_col] = pd.to_numeric(chunk[id_col], errors='coerce').astype('Int64')
    chunk = chunk.dropna(subset=[id_col])

    merged = chunk.merge(quint_map, left_on=id_col, right_on='transcriptid', how='inner')
    if merged.empty:
        continue

    # Ensure numeric returns
    for c in return_cols:
        merged[c] = pd.to_numeric(merged[c], errors='coerce')

    g = merged.groupby('div_quintile', observed=True)
    sums = g[return_cols].sum(min_count=1)
    cnts = g[return_cols].count()

    if sum_by_q is None:
        sum_by_q = sums
        cnt_by_q = cnts
    else:
        sum_by_q = sum_by_q.add(sums, fill_value=0)
        cnt_by_q = cnt_by_q.add(cnts, fill_value=0)

if sum_by_q is None:
    raise ValueError('No rows matched between event file and div_wide transcriptids. Check transcriptid alignment.')

mean_by_q = sum_by_q / cnt_by_q.replace(0, np.nan)
mean_by_q = mean_by_q.reset_index()

# Convert wide -> long for plotting
mean_long = mean_by_q.melt(id_vars=['div_quintile'], value_vars=return_cols,
                           var_name='event_col', value_name='mean_return')

def _col_to_tau(col: str) -> int | None:
    s = col.lower().replace(' ', '')
    s = s.replace('ret_t', 't').replace('r_t', 't')
    if s.startswith('t'):
        s = s[1:]
    if s == '0' or s == '':
        return 0
    try:
        return int(s)
    except Exception:
        return None

mean_long['tau'] = mean_long['event_col'].map(_col_to_tau)
mean_long = mean_long.dropna(subset=['tau']).sort_values(['div_quintile','tau'])

# Convert mean daily returns -> cumulative returns, indexed to 0 at t=0
mean_long['mean_return'] = pd.to_numeric(mean_long['mean_return'], errors='coerce')
mean_long['cum_return'] = mean_long.groupby('div_quintile', observed=True)['mean_return'].cumsum()

# Index at t=0: subtract cumulative value at tau==0 (so cum_return(0) == 0)
base0 = (
    mean_long.loc[mean_long['tau'] == 0, ['div_quintile','cum_return']]
            .rename(columns={'cum_return':'base0'})
)
mean_long = mean_long.merge(base0, on='div_quintile', how='left')
mean_long['cum_return_idx0'] = mean_long['cum_return'] - mean_long['base0'].fillna(0.0)

# Plot: cumulative event-study lines by divergence quintile
fig = go.Figure()
for q, d in mean_long.groupby('div_quintile', sort=False):
    fig.add_trace(go.Scatter(
        x=d['tau'], y=d['cum_return_idx0'],
        mode='lines+markers',
        name=str(q),
        line=dict(width=2),
        marker=dict(size=5),
        hovertemplate='Q: %{text}<br>Day: %{x}<br>Cumulative return: %{y:.5f}<extra></extra>',
        text=[str(q)] * len(d),
    ))

fig.add_vline(x=0, line_dash='dot', line_color=BORDER, line_width=1)
fig.add_hline(y=0, line_dash='dot', line_color=BORDER, line_width=1)

theme(fig,
      'Event Study: Cumulative Mean Return (Indexed at t=0) by Divergence Quintile',
      'Event time (days)',
      'Cumulative mean return (t=0 indexed)',
      h=520)
fig.show()


Event file columns:
['transcriptid', 'companyid', 'headline', 'transcriptcreationdate_utc', 'mostimportantdateutc', 'companyname', 'ticker', 'event_type', 'full_transcript_text', 'call_date', 'permno', 'actual_call_date', 'close_price_call_day', 'open_price_next_day', 'close_to_open_return', 'ret_t-15', 'ret_t-14', 'ret_t-13', 'ret_t-12', 'ret_t-11', 'ret_t-10', 'ret_t-9', 'ret_t-8', 'ret_t-7', 'ret_t-6', 'ret_t-5', 'ret_t-4', 'ret_t-3', 'ret_t-2', 'ret_t-1', 'ret_t0', 'ret_t1', 'ret_t2', 'ret_t3', 'ret_t4', 'ret_t5', 'ret_t6', 'ret_t7', 'ret_t8', 'ret_t9', 'ret_t10', 'ret_t11', 'ret_t12', 'ret_t13', 'ret_t14', 'ret_t15', 'transcript_length', 'word_count']

Sample rows:


,transcriptid,companyid,headline,transcriptcreationdate_utc,mostimportantdateutc,companyname,ticker,event_type,full_transcript_text,call_date,...,ret_t8,ret_t9,ret_t10,ret_t11,ret_t12,ret_t13,ret_t14,ret_t15,transcript_length,word_count
0,46981,18671,"Albemarle Corp., Q4 2009 Earnings Call, Jan 26...",2010-01-27,2010-01-26,Albemarle Corporation,ALB,Earnings Calls,"Good day, ladies and gentlemen, and welcome to...",2010-01-26,...,-0.006499,-0.008532,0.013196,0.001699,0.013850,0.020351,0.030328,0.001591,66674,11638
1,58004,18671,"Albemarle Corp., Q1 2010 Earnings Call, Apr-27...",2010-04-28,2010-04-27,Albemarle Corporation,ALB,Earnings Calls,"Good day, ladies and gentlemen and welcome to ...",2010-04-27,...,-0.038245,0.087178,-0.009144,0.028632,-0.008052,-0.026206,-0.006668,0.006234,50584,8758
2,69591,18671,"Albemarle Corp., Q2 2010 Earnings Call, Jul-27...",2010-07-27,2010-07-27,Albemarle Corporation,ALB,Earnings Calls,"Ladies and gentlemen, thank you for standing b...",2010-07-27,...,0.003093,0.010130,-0.019185,-0.038009,-0.025878,-0.010199,0.001917,0.025114,43225,7413
3,81269,18671,"Albemarle Corp., Q3 2010 Earnings Call, Oct 22...",2010-10-22,2010-10-22,Albemarle Corporation,ALB,Earnings Calls,"Good day, ladies and gentlemen, and welcome to...",2010-10-22,...,0.004729,0.035105,0.006063,-0.004143,-0.007186,0.009524,-0.000755,-0.018882,54159,9394
4,49605,18711,"Allstate Corp., Q4 2009 Earnings Call, Feb 11,...",2010-02-11,2010-02-11,The Allstate Corporation,ALL,Earnings Calls,"Good day, ladies and gentlemen, and welcome to...",2010-02-11,...,0.003207,0.000639,-0.001597,0.010240,0.014254,-0.000937,-0.001563,0.006262,55702,9586



Detected return-like columns:
['ret_t-15', 'ret_t-14', 'ret_t-13', 'ret_t-12', 'ret_t-11', 'ret_t-10', 'ret_t-9', 'ret_t-8', 'ret_t-7', 'ret_t-6', 'ret_t-5', 'ret_t-4', 'ret_t-3', 'ret_t-2', 'ret_t-1', 'ret_t0', 'ret_t1', 'ret_t2', 'ret_t3', 'ret_t4', 'ret_t5', 'ret_t6', 'ret_t7', 'ret_t8', 'ret_t9', 'ret_t10', 'ret_t11', 'ret_t12', 'ret_t13', 'ret_t14', 'ret_t15']

Will retain these columns (pending your confirmation):
['transcriptid', 'ret_t-15', 'ret_t-14', 'ret_t-13', 'ret_t-12', 'ret_t-11', 'ret_t-10', 'ret_t-9', 'ret_t-8', 'ret_t-7', 'ret_t-6', 'ret_t-5', 'ret_t-4', 'ret_t-3', 'ret_t-2', 'ret_t-1', 'ret_t0', 'ret_t1', 'ret_t2', 'ret_t3', 'ret_t4', 'ret_t5', 'ret_t6', 'ret_t7', 'ret_t8', 'ret_t9', 'ret_t10', 'ret_t11', 'ret_t12', 'ret_t13', 'ret_t14', 'ret_t15']


,transcriptid,ret_t-15,ret_t-14,ret_t-13,ret_t-12,ret_t-11,ret_t-10,ret_t-9,ret_t-8,ret_t-7,...,ret_t6,ret_t7,ret_t8,ret_t9,ret_t10,ret_t11,ret_t12,ret_t13,ret_t14,ret_t15
0,46981,0.047292,-0.001838,0.001841,-0.007088,0.015071,-0.004428,-0.013867,0.010082,-0.003940,...,-0.007317,-0.033852,-0.006499,-0.008532,0.013196,0.001699,0.013850,0.020351,0.030328,0.001591
1,58004,0.010658,-0.000229,0.001605,0.014423,-0.018055,-0.008504,0.031989,-0.000225,-0.013705,...,-0.029260,-0.054255,-0.038245,0.087178,-0.009144,0.028632,-0.008052,-0.026206,-0.006668,0.006234
2,69591,-0.002551,0.045269,0.014436,0.001688,-0.019022,0.037064,-0.010888,0.003589,-0.038865,...,0.024836,-0.002644,0.003093,0.010130,-0.019185,-0.038009,-0.025878,-0.010199,0.001917,0.025114
3,81269,0.009613,-0.011849,0.030835,-0.004570,-0.003548,0.001257,0.004811,-0.000416,0.016660,...,0.001197,0.011158,0.004729,0.035105,0.006063,-0.004143,-0.007186,0.009524,-0.000755,-0.018882
4,49605,-0.012492,-0.027571,0.012342,-0.002306,0.000000,-0.010898,-0.000668,0.000668,0.019032,...,0.006086,-0.007322,0.003207,0.000639,-0.001597,0.010240,0.014254,-0.000937,-0.001563,0.006262


In [ ]:
# ── Call-level signal table ────────────────────────────────────────────────────
# Merge LM + FinBERT + divergence into one call-level frame for downstream use

call_signals = div_wide[[
    'transcriptid','ticker','call_year','quarter','close_to_open_return',
    'mean_fb_net_ceo','mean_fb_net_cfo','mean_fb_neg_ceo','mean_fb_neg_cfo',
    'ceo_cfo_net_delta','ceo_cfo_abs_div','ceo_more_positive',
]].merge(
    lm_call_agg[lm_call_agg['role'] == 'CEO']
               .rename(columns={'mean_lm_net':'lm_net_ceo','mean_lm_unc':'lm_unc_ceo'})
               .drop(columns='role'),
    on=['transcriptid','ticker','call_year'], how='left',
).merge(
    lm_call_agg[lm_call_agg['role'] == 'CFO']
               .rename(columns={'mean_lm_net':'lm_net_cfo','mean_lm_unc':'lm_unc_cfo'})
               .drop(columns='role'),
    on=['transcriptid','ticker','call_year'], how='left',
)

SIGNALS_PATH = PROC_DIR / 'call_signals.parquet'
call_signals.to_parquet(SIGNALS_PATH, index=False)

print(f'Saved call-level signals: {len(call_signals):,} rows → {SIGNALS_PATH}')
print(call_signals.describe().round(4))

Saved call-level signals: 60 rows → ../exec_transcripts/processed/call_signals.parquet
       transcriptid  call_year  close_to_open_return  mean_fb_net_ceo  \
count  6.000000e+01     60.000               60.0000          60.0000   
mean   1.383155e+06   2017.000                0.0012           0.1816   
std    9.879206e+05      4.357                0.0089           0.0894   
min    4.847500e+04   2010.000               -0.0185           0.0210   
25%    5.268205e+05   2013.000               -0.0037           0.1158   
50%    1.219661e+06   2017.000                0.0013           0.1691   
75%    2.203734e+06   2021.000                0.0047           0.2464   
max    3.294948e+06   2024.000                0.0294           0.4866   

       mean_fb_net_cfo  mean_fb_neg_ceo  mean_fb_neg_cfo  ceo_cfo_net_delta  \
count          60.0000          60.0000          60.0000            60.0000   
mean            0.1499           0.0561           0.1004             0.0317   
std             0.

## 7 · Salience dictionary (weighted clusters)

**Goal:** score sentences with a **topic-clustered, weighted vocabulary** so you get (1) a total salience score and (2) a **cluster breakdown** for interpretability and tuning.

**Design notes:**
- **Multi-word terms first:** terms are matched longest-first and overlapping spans are not double-counted (e.g. `net interest margin` before `margin`).
- **Time-varying weight:** `emerging_tech` cluster can be boosted post-2022 (simple year multiplier).
- **Sector hook:** pass `sector='financials'` etc. to upweight that cluster for demos.
- **Temporal alignment (Layer 3):** this is *not* embedding-based pairwise diff across quarters; use `temporal_delta()` on cluster-level scores between two highlighted call outputs as a lightweight, interpretable delta.

**HTML view:** `highlight_call_html()` / `write_highlight_html()` render the full call with **speaker name**, role, section, and sentences tinted by **dominant cluster** (legend) and **intensity** vs. the strongest sentence on that call (hover for cluster breakdown).

**Latest call:** `latest_call_turns(df, ticker=...)` selects the rowset with **max `call_date`** (then max `transcriptid` if tied)—not the first `transcriptid` in sort order (which skews old).

The dictionary below is a starting prior — calibrate on a hand-read sample of calls before relying on weights for inference.

In [159]:
# ── 7 · Salience dictionary: import + smoke test ───────────────────────────────
# Module: assignments/salience_dictionary.py (works if cwd is repo root or assignments/)
import importlib
import sys
from pathlib import Path

for p in (Path.cwd(), Path.cwd() / "assignments"):
    if (p / "salience_dictionary.py").exists():
        sys.path.insert(0, str(p))
        break

import salience_dictionary as sal
importlib.reload(sal)

# Example sentence
ex = (
    "We're narrowing full-year guidance given margin compression and supply chain headwinds, "
    "but we remain confident in our long-term outlook."
)
tot, br = sal.score_sentence(ex, year=2024, sector=None)
print('Total:', tot)
print('Top clusters:', sorted(br.items(), key=lambda x: -x[1])[:5])

# Optional: latest call for one ticker → HTML highlight + top sentences (needs load_calls)
# SALIENCE_DEMO_TICKER = 'MA'
# if 'load_calls' in dir():
#     demo_turns = load_calls(
#         tickers=[SALIENCE_DEMO_TICKER],
#         cols=['transcriptid','componentorder','componenttext','transcriptpersonname',
#               'ticker','companyname','call_date','call_year','role','section'],
#     )
#     sub = sal.latest_call_turns(demo_turns, ticker=SALIENCE_DEMO_TICKER)
#     tid = int(sub['transcriptid'].iloc[0])
#     yr = int(sub['call_year'].iloc[0])
#     out_html = PROC_DIR / f'salience_highlight_{SALIENCE_DEMO_TICKER}_{tid}.html'
#     sal.write_highlight_html(out_html, sub, year=yr, sector=None)
#     print(f'Wrote {out_html.resolve()}')
#     hits = sal.highlight_transcript(sub, year=yr, sector=None, top_n=15)
#     print(f'\nTop salience sentences (transcriptid={tid}, year={yr}, latest call_date):')
#     for h in hits[:15]:
#         print(f"  [{h['top_cluster']}] {h['score']:.2f} | {h['sentence'][:120]}...")

Total: 20.9
Top clusters: [('guidance', 11.4), ('margin', 5.6), ('macro', 2.5), ('mgmt_signal', 1.4), ('capital', 0.0)]

Top salience sentences (transcriptid=48475, year=2010):
  [revenue] 8.08 | We achieved full year net revenue of $5.1 billion, representing growth of 2.1% and on a constant currency basis, net revenue grew 3.9%....
  [margin] 6.92 | Finally, we remain committed to our longer term of objectives of annual margin expansion of three to five percentage points and average net income growth of 20% to 30%....
  [revenue] 6.62 | On a local currency basis, worldwide purchase volume grew 5.7% and U.S. purchase volume declined 1.3% for the quarter driven by a declining credit volume....
  [guidance] 6.54 | We delivered strong financial results for the full year with net revenue growth of 2.1% on an as reported basis or almost 4% on a constant currency basis....
  [guidance] 6.00 | Quarter over quarter over quarter, it's probably the hardest line for us to forecast, as well as for